# Guardrails & Untrusted Content
Agents in production must be shielded from both malicious inputs (Prompt Injection/PII leaks) and restricted from generating harmful or unformatted outputs.

This notebook demonstrates SOTA Guardrail patterns:
1. **Pre-LLM Input Scrubbing:** Redacting PII *before* the token hits the expensive/untrusted LLM.
2. **Post-LLM Output Validation:** Forcing structural and semantic compliance using `pydantic`.

**Dependencies required:** `pip install pydantic presidio-analyzer presidio-anonymizer` (simulated below)


## 1. Pre-LLM Guardrails (PII Scrubbing)
Never send raw user data containing Social Security Numbers or Credit Cards to a third-party LLM provider like OpenAI. You must intercept and redact it locally first.


In [1]:
import re

def local_pii_scrubber(text: str) -> str:
    """
    A rudimentary local scrubber. In production, use Microsoft Presidio.
    This runs entirely on your local CPU before making an API call to the LLM.
    """
    # Redact Social Security Numbers
    text = re.sub(r'\b\d{3}-\d{2}-\d{4}\b', '[REDACTED_SSN]', text)
    # Redact standard Credit Card patterns
    text = re.sub(r'\b(?:\d{4}[ -]?){3}\d{4}\b', '[REDACTED_CC]', text)
    return text

raw_user_input = "Hi, my name is John. My SSN is 123-45-6789 and my card is 4111-2222-3333-4444. Can you refund me?"

print("🔴 RAW INPUT (Dangerous to send to OpenAI):")
print(raw_user_input)

safe_input = local_pii_scrubber(raw_user_input)
print("\n🟢 SCRUBBED INPUT (Safe for LLM Inference):")
print(safe_input)


🔴 RAW INPUT (Dangerous to send to OpenAI):
Hi, my name is John. My SSN is 123-45-6789 and my card is 4111-2222-3333-4444. Can you refund me?

🟢 SCRUBBED INPUT (Safe for LLM Inference):
Hi, my name is John. My SSN is [REDACTED_SSN] and my card is [REDACTED_CC]. Can you refund me?


## 2. Post-LLM Guardrails (Output Validation)
LLMs hallucinate formats and tone. SOTA architectures use `pydantic` validators (or frameworks like NeMo Guardrails) to strictly enforce output rules, automatically rejecting and retrying if the LLM fails the check.


In [2]:
from pydantic import BaseModel, Field, field_validator, ValidationError
import sys, os; sys.path.insert(0, os.path.join(os.getcwd(), 'curriculum/intermediate/04-guardrails-untrusted-content')); from policy import CustomerResponse
# --- Simulation 1: The LLM generates an unsafe response ---
mock_llm_bad_output = {
    "tone": "aggressive",
    "message": "We are much better than Competitor_A. Deal with it."
}
print("--- TESTING UNSAFE LLM OUTPUT ---")
try:
    validated_output = CustomerResponse(**mock_llm_bad_output)
except ValidationError as e:
    print("🛡️ Guardrail caught a violation! The agent is prevented from responding.")
    for error in e.errors():
        print(f"   - Failed Field '{error['loc'][0]}': {error['msg']}")
print("\n--- TESTING SAFE LLM OUTPUT ---")
mock_llm_good_output = {
    "tone": "professional",
    "message": "We apologize for the inconvenience and are working to resolve the issue."
}
try:
    validated_output = CustomerResponse(**mock_llm_good_output)
    print("✅ Guardrails passed. Output is safe to send to user:")
    print(f"   Tone: {validated_output.tone}")
    print(f"   Message: {validated_output.message}")
except ValidationError as e:
    print(e)


--- TESTING UNSAFE LLM OUTPUT ---
🛡️ Guardrail caught a violation! The agent is prevented from responding.
   - Failed Field 'tone': Value error, Guardrail triggered: Unacceptable tone 'aggressive'. Must be polite, professional, or empathetic.
   - Failed Field 'message': Value error, Guardrail triggered: Message contains a mention of a competitor.

--- TESTING SAFE LLM OUTPUT ---
✅ Guardrails passed. Output is safe to send to user:
   Tone: professional
   Message: We apologize for the inconvenience and are working to resolve the issue.


## 3. The Guardrail Retry Loop
When a guardrail fails, you don't just crash. You feed the `ValidationError` message back to the LLM and ask it to try again.


In [3]:
def guardrail_retry_loop(initial_llm_attempt: dict):
    max_retries = 3
    current_attempt = initial_llm_attempt
    
    for attempt in range(1, max_retries + 1):
        try:
            print(f"\nAttempt {attempt}: Validating...")
            valid_response = CustomerResponse(**current_attempt)
            print("✅ Success!")
            return valid_response
        except ValidationError as e:
            print(f"❌ Failed: {e.errors()[0]['msg']}")
            print("   -> Feeding error back to LLM for correction...")
            
            # MOCK LLM CORRECTION: We pretend the LLM fixed its mistake based on the error
            if "tone" in e.errors()[0]['loc']:
                current_attempt["tone"] = "polite"
            elif "message" in e.errors()[0]['loc']:
                current_attempt["message"] = "We are the best. Thank you."

    print("🚨 Max retries reached. Escalate to human.")
    return None

# Start the loop with a bad response
guardrail_retry_loop({
    "tone": "snarky",
    "message": "Go buy from competitor_a if you don't like it."
})



Attempt 1: Validating...
❌ Failed: Value error, Guardrail triggered: Unacceptable tone 'snarky'. Must be polite, professional, or empathetic.
   -> Feeding error back to LLM for correction...

Attempt 2: Validating...
❌ Failed: Value error, Guardrail triggered: Message contains a mention of a competitor.
   -> Feeding error back to LLM for correction...

Attempt 3: Validating...
✅ Success!


CustomerResponse(tone='polite', message='We are the best. Thank you.')